In [27]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [28]:
df=pd.read_csv("../../data/raw/major_leagues v1/sofascore_major_leagues_2425season.csv")
df_yearly=pd.read_csv("../../data/raw/major_leagues v1/sofascore_MLS_Argentina_25season.csv")

In [29]:
df.shape

(7979, 117)

In [30]:
df_yearly.shape

(1663, 117)

In [31]:
set(df.columns) == set(df_yearly.columns)

True

In [32]:
df.columns.tolist() == df_yearly.columns.tolist()

True

In [33]:
total_df = pd.concat([df, df_yearly], ignore_index=True)

In [34]:
total_df.shape

(9642, 117)

In [35]:
total_df = total_df.rename(columns={col: f"{col}".lower() for col in total_df.columns})

In [36]:
total_df=total_df.drop(columns=["outfielderblocks","totwappearances","goalsprevented"])

In [37]:
total_df["goals_minus_xg"] = total_df["goals"] - total_df["expectedgoals"]
total_df["assists_minus_xa"] = total_df["assists"] - total_df["expectedassists"]

C:\Users\vibha\AppData\Local\Temp\ipykernel_12312\2912888129.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["goals_minus_xg"] = total_df["goals"] - total_df["expectedgoals"]
C:\Users\vibha\AppData\Local\Temp\ipykernel_12312\2912888129.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["assists_minus_xa"] = total_df["assists"] - total_df["expectedassists"]


In [38]:
per90_cols = [
    # Finishing
    "goals",
    "expectedgoals",
    "goals_minus_xg",
    "totalshots",
    "shotsontarget",
    "shotsofftarget",
    "shotsfrominsidethebox",
    "shotsfromoutsidethebox",
    "hitwoodwork",
    "leftfootgoals",
    "rightfootgoals",
    "headedgoals",
    "goalsfrominsidethebox",
    "goalsfromoutsidethebox",
    "freekickgoal",
    "penaltiestaken",
    "penaltygoals",
    "bigchancesmissed",

    # Creativity
    "assists",
    "expectedassists",
    "assists_minus_xa",
    "goalsassistssum",
    "totalpasses",
    "accuratepasses",
    "inaccuratepasses",
    "totaloppositionhalfpasses",
    "accurateoppositionhalfpasses",
    "totalownhalfpasses",
    "accurateownhalfpasses",
    "accuratefinalthirdpasses",
    "keypasses",
    "totalattemptassist",
    "passtoassist",
    "bigchancescreated",
    "totallongballs",
    "accuratelongballs",
    "totalchippedpasses",
    "accuratechippedpasses",
    "totalcross",
    "accuratecrosses",

    # Possession
    "touches",
    "possessionlost",
    "dispossessed",
    "totalcontest",
    "successfuldribbles",
    "offsides",
    "wasfouled",
    "fouls",

    # Defensive
    "tackles",
    "tackleswon",
    "interceptions",
    "ballrecovery",
    "possessionwonattthird",
    "clearances",
    "blockedshots",
    "dribbledpast",
    "errorleadtoshot",
    "errorleadtogoal",

    # Duels
    "totalduelswon",
    "duellost",
    "groundduelswon",
    "aerialduelswon",
    "aeriallost",
    
    # Goalkeeping
    "saves",
    "savescaught",
    "savesparried",
    "savedshotsfrominsidethebox",
    "savedshotsfromoutsidethebox",
    "runsout",
    "successfulrunsout",
    "goalkicks",
    "punches",
    "highclaims",
    "crossesnotclaimed",  
    

    # Discipline
    "yellowcards",
    "yellowredcards",
    "directredcards",
    "owngoals"
]

rate_cols = [
    "goalconversionpercentage",
    "scoringfrequency",

    "accuratepassespercentage",
    "accuratelongballspercentage",
    "successfuldribblespercentage",

    "tackleswonpercentage",

    "totalduelswonpercentage",
    "groundduelswonpercentage",
    "aerialduelswonpercentage",

    "penaltyconversion",

    "rating",
    "totalrating",
    "countrating"
]

for col in per90_cols:
    total_df[f"{col}_per90"] = np.where(
        total_df["minutesplayed"] > 0,
        (total_df[col] / total_df["minutesplayed"]) * 90,
        np.nan
    )

total_df["goals_per_xg"] = np.where(
    total_df["expectedgoals"] > 0,
    total_df["goals"] / total_df["expectedgoals"],
    1
)

total_df["shots_on_target_pct"] = np.where(
    total_df["totalshots"] > 0,
    total_df["shotsontarget"] / total_df["totalshots"],
    0
)


total_df["inside_box_shot_pct"] = np.where(
    total_df["totalshots"] > 0,
    total_df["shotsfrominsidethebox"] / total_df["totalshots"],
    0
)

total_df["assist_conversion"] = np.where(
    total_df["keypasses"] > 0,
    total_df["assists"] / total_df["keypasses"],
    0
)


total_df["xa_per_keypass"] = np.where(
    total_df["keypasses"] > 0,
    total_df["expectedassists"] / total_df["keypasses"],
    0
)


total_df["final_third_pass_pct"] = np.where(
    total_df["accuratepasses"] > 0,
    total_df["accuratefinalthirdpasses"] / total_df["accuratepasses"],
    0
)

total_df["opp_half_pass_pct"] = np.where(
    total_df["accuratepasses"] > 0,
    total_df["accurateoppositionhalfpasses"] / total_df["accuratepasses"],
    0
)

total_df["dribbles_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["successfuldribbles"] / total_df["touches"],
    0
)

total_df["dispossessed_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["dispossessed"] / total_df["touches"],
    0
)

total_df["possession_lost_per_touch"] = np.where(
    total_df["touches"] > 0,
    total_df["possessionlost"] / total_df["touches"],
    0
)

total_df["defensive_actions"] = (
    total_df["tackles"] +
    total_df["interceptions"] +
    total_df["ballrecovery"]
)

total_df["defensive_actions_per90"] = (
    total_df["tackles_per90"] +
    total_df["interceptions_per90"] +
    total_df["ballrecovery_per90"]
)

C:\Users\vibha\AppData\Local\Temp\ipykernel_12312\3909454853.py:118: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f"{col}_per90"] = np.where(
C:\Users\vibha\AppData\Local\Temp\ipykernel_12312\3909454853.py:124: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df["goals_per_xg"] = np.where(
C:\Users\vibha\AppData\Local\Temp\ipykernel_12312\3909454853.py:130: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joini

In [39]:
total_df=total_df[total_df['minutesplayed'] > 900]

In [40]:
total_df['league'].value_counts()

league
USA MLS                       478
England EFL Championship      451
Spain La Liga 2               394
Italy Serie B                 353
Italy Serie A                 346
Spain La Liga                 341
England Premier League        320
Turkiye Super Lig             310
Germany 2.Bundesliga          289
France Ligue 1                288
France Ligue 2                287
Portugal Primeira Liga        286
Germany Bundesliga            278
Netherlands Eredivisie        277
Saudi Arabia Pro League       276
Argentina Liga Profesional    231
Name: count, dtype: int64

In [41]:
total_df.shape

(5205, 206)

In [42]:
metadata_features = [
    "player",
    "team",
    "player id",
    "team id",
    "league",
    "position",
    "minutesplayed",
    "matchesstarted",
    "appearances"
]

In [43]:
total_df=total_df.fillna(0.0)

In [44]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    null_values=total_df.isna().sum().sum()
    print(null_values)

0


In [45]:
total_rows = len(total_df)
unique_ids = total_df['player id'].nunique()
duplicate_count = total_rows - unique_ids
duplicate_count

32

In [46]:
features_to_scale = [
    col for col in total_df.columns 
    if total_df[col].dtype in [np.float64, np.int64] 
    and col not in metadata_features
]

for col in features_to_scale:
    grouped = total_df.groupby(["league", "position"])[col]
    group_mean = grouped.transform("mean")
    group_std = grouped.transform("std")
    
    total_df[f"{col}_zscore"] = np.where(
        group_std > 0,
        (total_df[col] - group_mean) / group_std,
        0.0
    )

C:\Users\vibha\AppData\Local\Temp\ipykernel_12312\1788466843.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  total_df[f"{col}_zscore"] = np.where(


In [47]:
total_df.columns.to_list()

['accuratechippedpasses',
 'accuratecrosses',
 'accuratecrossespercentage',
 'accuratefinalthirdpasses',
 'accuratelongballs',
 'accuratelongballspercentage',
 'accurateoppositionhalfpasses',
 'accurateownhalfpasses',
 'accuratepasses',
 'accuratepassespercentage',
 'aerialduelswon',
 'aerialduelswonpercentage',
 'aeriallost',
 'appearances',
 'assists',
 'attemptpenaltymiss',
 'attemptpenaltypost',
 'attemptpenaltytarget',
 'ballrecovery',
 'bigchancescreated',
 'bigchancesmissed',
 'blockedshots',
 'cleansheet',
 'clearances',
 'countrating',
 'crossesnotclaimed',
 'directredcards',
 'dispossessed',
 'dribbledpast',
 'duellost',
 'errorleadtogoal',
 'errorleadtoshot',
 'expectedassists',
 'expectedgoals',
 'fouls',
 'freekickgoal',
 'goalconversionpercentage',
 'goalkicks',
 'goals',
 'goalsassistssum',
 'goalsconceded',
 'goalsconcededinsidethebox',
 'goalsconcededoutsidethebox',
 'goalsfrominsidethebox',
 'goalsfromoutsidethebox',
 'groundduelswon',
 'groundduelswonpercentage',
 'h

In [48]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(total_df[total_df.duplicated(subset=['player id'], keep=False)].drop_duplicates(subset=['player id'])[['player id', 'player']])

,player id,player
20,796143,Alex Palmer
64,1013842,Emmanuel Agbadou
335,894733,Marshall Munetsi
460,873554,Omar Marmoush
463,111505,Son Heung-min
528,361352,Adam Armstrong
575,253809,Rui Silva
794,862464,Antonio Candela
821,249399,Rodrigo De Paul
914,857178,Kike Pérez


In [49]:
with pd.option_context('display.max_rows', None, 'display.max_colwidth', None):
    display(total_df[['team', 'team id']].drop_duplicates().dropna())

,team,team id
3,Brentford,50
5,Everton,48
8,Bournemouth,60
9,Nottingham Forest,14
10,Southampton,45
13,Newcastle United,39
14,Ipswich Town,32
16,Liverpool FC,44
17,West Ham United,37
18,Tottenham Hotspur,33


In [50]:
player_team_ids = {
    796143:   32,     # Alex Palmer           → Ipswich Town (from West Brom Feb 3, ended season there)
    1013842:  3,      # Emmanuel Agbadou      → Wolverhampton (from Reims Jan 9, ended season there)
    894733:   3,      # Marshall Munetsi      → Wolverhampton (from Reims Feb 3, ended season there)
    873554:   17,     # Omar Marmoush         → Manchester City (from Frankfurt Jan 2025)
    111505:   33,     # Son Heung-min         → Tottenham Hotspur (left Aug 2025)
    361352:   45,     # Adam Armstrong        → Southampton (Championship, left Feb 2026)
    253809:   3001,   # Rui Silva             → Sporting CP (loan from Betis Jan 14, ended season there)
    862464:   2831,   # Antonio Candela       → Real Valladolid (loan from Venezia Feb 3, ended season there)
    249399:   337602, # Rodrigo De Paul       → Inter Miami CF (2025 MLS season, from Atlético Madrid Jul 2025)
    857178:   2688,   # Kike Pérez            → Venezia (from Valladolid Jan 29, ended season there)
    991479:   1658,   # Anthony Rouault       → Stade Rennais (from Stuttgart Feb 3, ended season there)
    879746:   2569,   # Leo Østigård          → TSG Hoffenheim (loan from Rennes Feb 3, ended season there)
    930272:   2542,   # Tom Krauß             → VfL Bochum (loan from Mainz, recalled from Luton Jan 29)
    35166:    22010,  # Thomas Müller         → Vancouver Whitecaps (2025 MLS season, from Bayern Aug 2025)
    149728:   5885,   # Budu Zivzivadze       → 1. FC Heidenheim (from Karlsruhe Jan 3, ended season there)
    812829:   2715,   # Giangiacomo Magnani   → Palermo (from Hellas Verona Jan 28, ended season there)
    889259:   1644,   # Khvicha Kvaratskhelia → Paris Saint-Germain (from Napoli Jan 18)
    137155:   2715,   # Joel Pohjanpalo       → Palermo (from Venezia Feb 3, ended season there)
    1445625:  1658,   # Jérémy Jacquet        → Stade Rennais (recalled from Clermont Feb 2025)
    858977:   1676,   # Christopher Operi     → Amiens SC (Ligue 2, stayed all season)
    1003400:  1682,   # Mory Gbane            → Stade de Reims (from Gil Vicente Jan 2025)
    889776:   21,     # Ryan Porteous         → Preston North End (loan from Watford Feb 3, ended season there)
    797691:   2561,   # Marvin Mehlem         → SC Paderborn 07 (loan from Hull City Jan 3, ended season there)
    848074:   243211, # Emmanuel Latte Lath   → Atlanta United (2025 MLS season)
    962411:   2846,   # Nicolás Fernández     → Elche (La Liga 2, left for NYCFC Jul 2025)
    910033:   3014,   # Cedric Teguia         → Moreirense (from Celta B Feb 3, ended season there)
    895533:   377973, # Myrto Uzuni           → Austin FC (2025 MLS season, from Granada Jan 24)
    915129:   2512,   # Osaze Urhoghide       → FC Dallas (2025 MLS season, joined Jan)
    822703:   2511,   # Mamadou Fofana        → New England Revolution (2025 MLS season, joined Jan)
    845754:   34469,  # Wenderson Galeno      → Al-Ahli (from Porto Jan 31, ended Saudi season there)
    862194:   56023,  # Zaydou Youssouf       → Al-Fateh (from Famalicão Jan 30, ended Saudi season there)
    1106549:  337602, # Telasco Segovia       → Inter Miami CF (2025 MLS season, from Casa Pia Jan 17)
}

In [51]:
for player_id, target_team_id in player_team_ids.items():
    player_rows = total_df[total_df["player id"] == player_id]
    if len(player_rows) <= 1:
        continue
        
    total_mins = player_rows["minutesplayed"].sum()
    if total_mins == 0:
        continue
        
    target_idx = player_rows[player_rows["team id"] == target_team_id].index
    if len(target_idx) == 0:
        target_idx = [player_rows.index[0]]
        
    keep_idx = target_idx[0]
    drop_indices = player_rows.index[player_rows.index != keep_idx]
    
    total_df.loc[keep_idx, "minutesplayed"] = total_mins
    
    for col in total_df.columns:
        if col in metadata_features and col != "minutesplayed":
            continue
            
        if col.endswith("_zscore"):
            weighted_sum = (player_rows["minutesplayed"] * player_rows[col]).sum()
            total_df.loc[keep_idx, col] = weighted_sum / total_mins
        elif col.endswith("_per90") or col in per90_cols:
            weighted_sum = (player_rows["minutesplayed"] * player_rows[col]).sum()
            if total_df[col].dtype in [np.int64, np.int32]:
                total_df = total_df.astype({col: "float64"})
            total_df.loc[keep_idx, col] = weighted_sum / total_mins
        elif col in rate_cols or col in ["goals_per_xg", "shots_on_target_pct", "inside_box_shot_pct", "assist_conversion", "xa_per_keypass", "final_third_pass_pct", "opp_half_pass_pct", "dribbles_per_touch", "dispossessed_per_touch", "possession_lost_per_touch", "weak_foot_goals_pct"]:
            weighted_sum = (player_rows["minutesplayed"] * player_rows[col]).sum()
            if total_df[col].dtype in [np.int64, np.int32]:
                total_df = total_df.astype({col: "float64"})
            total_df.loc[keep_idx, col] = weighted_sum / total_mins
            
    total_df = total_df.drop(drop_indices)

total_df = total_df.reset_index(drop=True)

In [52]:
total_df.to_csv("../../data/processed/major_leagues/Sofascore_player_data_2425.csv",index=False)